In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scrublet as scr
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rcParams
from matplotlib import rc
from matplotlib import colors
import seaborn as sb
from collections import Counter
import glob

sc.settings.set_figure_params(dpi = 160, color_map = 'RdPu', dpi_save = 300, format = 'pdf')
plt.rcParams["figure.figsize"] = [6,6]

In [2]:
sb.set_palette('colorblind')

In [3]:
# set working directory
%cd /nfs/team205/cs42/HSC/HSC_new_to_transfer/

/nfs/team205/cs42/HSC/HSC_new_to_transfer


In [4]:
from matplotlib import font_manager
font_manager.fontManager.addfont("csv/Helvetica.ttf")
print(font_manager.findfont("Helvetica"))
plt.rcParams["font.sans-serif"] = ["Helvetica"]
plt.rcParams["pdf.fonttype"] = 42

plt.rcParams['savefig.dpi'] = 300

plt.rcParams["axes.grid"] = False
plt.rcParams["pdf.fonttype"] = 42

/nfs/team205/cs42/HSC/HSC_new_to_transfer/csv/Helvetica.ttf


In [5]:
# figure directory
fig_path = 'figure/'
sc.settings.figdir = fig_path

# Download dataset

In [6]:
manifest = pd.read_csv('manifest/sample_manifest_23062026_manuscript.csv')
manifest = manifest.loc[manifest['screen_no']=='HOM_1',:]

In [7]:
smp_dict = list(manifest['sequence_name'])

In [8]:
smp_dict

['6464STDY14797954', '6464STDY14797955']

In [9]:
# check whether all files have been downloaded in matrix folder
DATAFOLDER = '/nfs/team205/cs42/HSC_matrix/'
exist = [x.split('/')[-1] for x in glob.glob(DATAFOLDER+'*')]

if len([x for x in smp_dict if x not in exist])>0:
    print([smp_dict[x][2] for x in smp_dict if x not in exist],'not in matrix folder')

In [14]:
# max_n_count is decided from the web_summary.html plot - the second step is around 200-400 where soup begins 
def read_files(filename, raw_file_path = '/nfs/team205/cs42/HSC_matrix/',
               max_n_count = 200):

    
    path = '%s/%s/'%(raw_file_path,filename)
    adata = sc.read_10x_h5(path+'raw_feature_bc_matrix.h5', gex_only=False)
    
    adata.obs_names = [filename+"-"+x.strip("-1") for x in adata.obs_names]
   
    # define GEX and antibody_capture
    gex = np.array(adata.var['feature_types']=='Gene Expression')
    antibody_capture = np.array(adata.var['feature_types']=='Antibody Capture')
    
    # caculate n_counts / n_genes per cell
    adata.obs['n_counts_GEX'] = np.sum(adata[:,gex].X, axis=1).A1
    adata.obs['n_genes'] = np.sum(adata[:,gex].X>0,axis=1)
    adata.obs['n_counts_antibody_capture'] = np.sum(adata[:,antibody_capture].X, axis=1).A1
    
    # filter cells
    clist = []
    clist.append(np.array(adata.obs['n_counts_GEX'] < max_n_count))
    
    c = np.column_stack(clist).all(axis=1)
    adata = adata[c].copy()
    
    # update manifest onto .obs
    adata.obs['file'] = filename
    adata.obs['sanger_sample_ID'] = manifest.loc[manifest['sequence_name']==filename, 'sanger_sample_ID'].item()
    adata.obs['HSC_days'] = manifest.loc[manifest['sequence_name']==filename, 'HSC days'].item()
    adata.obs['FACS_sort'] = manifest.loc[manifest['sequence_name']==filename, 'FACS_sort'].item()
    
    sc.write('%s_soup'%(filename),adata)

In [15]:
from multiprocessing import Pool

with Pool(10) as p:
    p.map(read_files, [x for x in smp_dict])

/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


# Merge dataset

In [16]:
ad = {}

In [17]:
for filename in smp_dict:
    ad[filename] = sc.read('%s_soup'%(filename))
    ad[filename].var_names_make_unique()

/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [18]:
merged = None;
for key in ad: 
    if merged is None:
        merged = ad[key]
    else:
        merged = merged.concatenate(ad[key], join='outer')

/tmp/ipykernel_1294208/1590458944.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged = merged.concatenate(ad[key], join='outer')


In [19]:
Counter(merged.obs['file'])

Counter({'6464STDY14797954': 672180, '6464STDY14797955': 623506})

# Call TF based on barcode

In [20]:
exog_genes = merged.var_names[merged.var_names.str.startswith('EXOG_')]
exog_exception = ['EXOG_mApple','EXOG_Bsd', 'EXOG_Bxb1','EXOG_PuroR']
exog_genes_count = [gene for gene in exog_genes if gene not in exog_exception]

In [21]:
exog_genes_count

['EXOG_EGFP',
 'EXOG_GFI1B',
 'EXOG_GFI1',
 'EXOG_HLF',
 'EXOG_IKZF1',
 'EXOG_IRF5',
 'EXOG_MYB_CON',
 'EXOG_MYCN',
 'EXOG_NFE2',
 'EXOG_RUNX1',
 'EXOG_SP100variant1',
 'EXOG_SPI1',
 'EXOG_TFEC',
 'EXOG_BCL11Avariant1']

In [31]:
from scipy import sparse

q99_by_dataset = {}

for key in ad.keys():
    adata = ad[key]

    # Optional: keep only genes present in this dataset
    genes = [g for g in exog_genes_count if g in adata.var_names]

    X = adata[:, genes].X

    q99_values = []

    if sparse.issparse(X):
        X = X.tocsc()  # efficient column/gene access

        for j in range(X.shape[1]):
            vals = X.data[X.indptr[j]:X.indptr[j + 1]]  # non-zero values for gene j

            if vals.size == 0:
                q99_values.append(np.nan)
            else:
                q99_values.append(np.percentile(vals, 95))

    else:
        X = np.asarray(X)

        for j in range(X.shape[1]):
            vals = X[:, j]
            vals = vals[vals != 0]

            if vals.size == 0:
                q99_values.append(np.nan)
            else:
                q99_values.append(np.percentile(vals, 95))

    q99_by_dataset[key] = pd.Series(q99_values, index=genes, name=key)

# Wide dataframe: rows = genes, columns = datasets
q99_df = pd.concat(q99_by_dataset.values(), axis=1)

q99_df

,6464STDY14797954,6464STDY14797955
EXOG_EGFP,1.0,1.0
EXOG_GFI1B,1.0,1.0
EXOG_GFI1,1.0,1.0
EXOG_HLF,1.0,1.0
EXOG_IKZF1,NaN,NaN
EXOG_IRF5,NaN,NaN
EXOG_MYB_CON,NaN,NaN
EXOG_MYCN,NaN,NaN
EXOG_NFE2,1.0,1.0
EXOG_RUNX1,NaN,NaN


In [22]:
df_plot = pd.DataFrame()
for key in ad:
    adata = ad[key]
    X = adata[:, exog_genes_count].X
    
    # extract non0 entries for plotting
    df = pd.DataFrame({'soup_non0_exp': X.data,
                      'sample': key})
    if df_plot.empty:
        df_plot = df
    else: 
        df_plot = pd.concat([df_plot, df], axis=0)
        
    print(key, 'non-zero%: ', X.nnz/X.shape[0]*100, 'max count: ', np.max(X),'99.9th centile: ',np.percentile(X.data, 99.9))
    print((X > 5).sum() / X.shape[0])

6464STDY14797954 non-zero%:  0.22984914754976346 max count:  3.0 99.9th centile:  2.0
0.0
6464STDY14797955 non-zero%:  0.03624664397776445 max count:  2.0 99.9th centile:  1.7750092
0.0


In [34]:
Counter(df_plot['soup_non0_exp'])

Counter({1.0: 1758, 2.0: 12, 3.0: 1})